In [ ]:
import ROOT
import numpy as np
import pandas as pd

#funzione modello
def model(x, A, B):
    return A * x + B

def model_derivative(x, A, B):
    dx=10e-6
    return (model(x + dx, A, B) - model(x, A, B)) / dx

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Sheet1")
variable = df.dropna(subset=["var1", "var2", "err_y"])
variable = variable.to_numpy()
df = df.to_numpy()

#valori iniziali dei parametri
A = 1
B = 2
C = 3

#definizione variabili
x = variable[:, 0]
y = variable[:, 1]

err_y = variable[:, 2]
err_x = df.iloc[0]['err_x'] #err_x costante,
err_x = np.full_like(x, err_x) #creo un array di err_x costante
err_yeff = np.sqrt(err_y**2 + (model_derivative(x, A, B) * err_x)**2) #errore effettivo

#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(x), max(x), 3)
model_fit.SetParameters(A, B, C)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr1.SetTitle("Dati con errori; x; y")
gr1.SetMarkerStyle(20)
gr1.SetMarkerColor(ROOT.kBlue)
gr1.SetLineColor(ROOT.kBlack)

fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = gr1.GetParameters()
errors = gr1.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"A = {parameters[0]} ± {errors[0]}")
print(f"B = {parameters[1]} ± {errors[1]}")
print(f"C = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("APX")

c1.Draw()
c1.SaveAs("fit_root.png")

In [ ]:
residuals_norm = (y - model(x, *fit_res.Parameters())) / err_yeff

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(x), max(x), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "#mu", "#sigma")
gaussian_fit.SetLineColor(ROOT.kGreen+2)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kBlue)
hist.SetFillColor(ROOT.kBlue)
hist.GetXaxis().SetTitle("valore")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = hist.GetParameters()
errors = hist.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

c2.Draw()
c2.SaveAs("hist.png")

In [ ]:
def line(x, m, q):
    return m * x + q

residuals = y - model(x, *fit_res.Parameters())

line_fit = ROOT.TF1('line', line, min(x), max(x), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")

gr2 = ROOT.TGraphErrors(len(x), x, residuals, np.zeros_like(residuals), err_yeff)
gr2.SetTitle("Dati con errori; x; y")
gr2.SetMarkerStyle(20)
gr2.SetMarkerColor(ROOT.kBlue)
gr2.SetLineColor(ROOT.kBlack)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = gr2.GetParameters()
errors = gr2.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")


c3.Draw()
c3.SaveAs("fit_line.png")